# FermaTwin model training
The IndPenSim benchmark (verified, published data: the reported accuracy) and Model A for the simulated yeast twin.

Attach the dataset with `yeast_batches.csv`, `process.csv` and `100_Batches_IndPenSim_Statistics.csv`, then **Run All**. Outputs land in `/kaggle/working/`.


## Model A — yeast run (simulated batches)

Trained on batches from the yeast simulation that drives the dashboard twin. Not a verified dataset: used for the what-if demo only; no accuracy claim is made from it.

In [ ]:
"""Model A: early final-protein prediction, fault detection, and drivers for the yeast run.

Runs locally (python ml/train_model_a.py) or as a Kaggle notebook with yeast_batches.csv attached as a dataset.
Writes model_a.json (consumed by the dashboard in the browser) and model_a_report.json (metrics for the cards).

Yield model: gradient-boosted trees on features that use only rows up to the prediction time (current value and change
over the last two hours of each signal, fraction of the batch elapsed, feedstock, operator setpoints). Split by batch.
80% interval: split-conformal, per batch stage, from out-of-fold residuals.

Fault model: multivariate statistical process control fitted on normal-window batches only. Each signal is centred and
scaled by its normal mean and spread at that minute of the batch; PCA on those residuals gives Hotelling T2 and SPE.
The score is the larger of the two, each divided by its 99th percentile on training normals. An alarm needs two
consecutive samples above the threshold.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold

SEED = 7
SIGNALS = ["dissolved_oxygen", "ph", "temperature", "agitation", "aeration", "feed_rate", "biomass", "co2", "substrate",
           "ethanol"]
SETPOINTS = ["sp_ph", "sp_temperature", "sp_agitation", "sp_aeration", "sp_feed"]
FEEDSTOCKS = ["glucose", "molasses", "pea-starch-hydrolysate"]
LAG_ROWS = 12  # two hours at the 10-minute log interval
TRAIN_FRACS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
REPORT_FRACS = [0.2, 0.4, 0.6, 0.8]


def find_data():
    for p in [Path("/kaggle/input"), Path('.').resolve().parent / "data" if "__file__" in globals() else Path(".")]:
        hits = list(p.rglob("yeast_batches.csv")) if p.exists() else []
        if hits:
            return hits[0]
    raise FileNotFoundError("yeast_batches.csv not found")


def out_dir():
    k = Path("/kaggle/working")
    if k.exists():
        return k
    d = Path('.').resolve().parent / "out"
    d.mkdir(exist_ok=True)
    return d


def feature_names():
    names = ["frac"] + SIGNALS + [f"d_{s}" for s in SIGNALS] + SETPOINTS + [f"fs_{f}" for f in FEEDSTOCKS]
    return names


def features_at(g, frac):
    """One row from a batch using only samples logged up to `frac` of the batch."""
    upto = g[g["frac"] <= frac + 1e-9]
    now = upto.iloc[-1]
    before = upto.iloc[max(0, len(upto) - 1 - LAG_ROWS)]
    row = {"frac": frac}
    for s in SIGNALS:
        row[s] = now[s]
        row[f"d_{s}"] = now[s] - before[s]
    for s in SETPOINTS:
        row[s] = now[s]
    for f in FEEDSTOCKS:
        row[f"fs_{f}"] = float(now["feedstock"] == f)
    return row, float(now["protein_kg"])


def build_table(df, fracs):
    rows, meta = [], []
    for b, g in df.groupby("batch", sort=False):
        g = g.sort_values("t_min")
        first = g.iloc[0]
        for f in fracs:
            r, so_far = features_at(g, f)
            rows.append(r)
            meta.append({"batch": b, "set": first["set"], "scenario": first["scenario"], "fault": int(first["fault"]),
                         "frac": f, "so_far": so_far, "y": float(first["final_protein_kg"])})
    return pd.DataFrame(rows)[feature_names()], pd.DataFrame(meta)


def gbr():
    return GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, min_samples_leaf=10,
                                     subsample=0.8, random_state=SEED)


def export_trees(model, names):
    trees = []
    for est in model.estimators_[:, 0]:
        t = est.tree_
        trees.append({
            "f": t.feature.tolist(), "t": [round(float(x), 6) for x in t.threshold],
            "l": t.children_left.tolist(), "r": t.children_right.tolist(),
            "v": [round(float(x), 6) for x in t.value[:, 0, 0]],
        })
    return {"features": names, "init": float(model.init_.constant_[0][0]), "lr": model.learning_rate, "trees": trees}


def mape(y, p):
    return float(np.mean(np.abs(p - y) / np.abs(y)) * 100)


def split_batches(batch_meta):
    rng = np.random.default_rng(SEED)
    test = set()
    for s in ["wide", "normal", "fault"]:
        ids = batch_meta.loc[batch_meta["set"] == s, "batch"].to_numpy()
        test |= set(rng.choice(ids, size=int(0.2 * len(ids)), replace=False).tolist())  # 80/20 by batch
    return test


# ---------------------------------------------------------------- fault detector (batch MSPC)

def fit_mspc(df_normal):
    piv = {s: df_normal.pivot(index="batch", columns="t_min", values=s) for s in SIGNALS}
    t_grid = piv[SIGNALS[0]].columns.to_numpy()
    mean = np.stack([piv[s].mean().to_numpy() for s in SIGNALS], axis=1)  # [T, S]
    std = np.stack([piv[s].std().to_numpy() for s in SIGNALS], axis=1)
    std = np.maximum(std, np.array([1.0, 0.02, 0.08, 2, 0.01, 0.003, 0.2, 0.05, 0.15, 0.15]))  # sensor floors
    return t_grid, mean, std


def residuals(df, t_grid, mean, std):
    idx = np.searchsorted(t_grid, df["t_min"].to_numpy()).clip(0, len(t_grid) - 1)
    x = df[SIGNALS].to_numpy()
    return (x - mean[idx]) / std[idx]


def mspc_scores(z, pca, t2_lim, spe_lim):
    scores = pca.transform(z)
    t2 = np.sum(scores ** 2 / pca.explained_variance_, axis=1)
    spe = np.sum((z - pca.inverse_transform(scores)) ** 2, axis=1)
    return np.maximum(t2 / t2_lim, spe / spe_lim)


def alarm_time(times, score, thr, start=60):
    """First time two consecutive samples exceed thr (after the first hour of start-up)."""
    over = (score > thr) & (times >= start)
    hit = np.where(over[1:] & over[:-1])[0]
    return float(times[hit[0] + 1]) if len(hit) else None


def main():
    df = pd.read_csv(find_data())
    out = out_dir()
    print(f"{df['batch'].nunique()} batches, {len(df)} rows")
    batch_meta = df.groupby("batch").first().reset_index()
    test_ids = split_batches(batch_meta)
    is_test = df["batch"].isin(test_ids)

    # ---------------- yield model, trained on non-fault batches
    train_df = df[~is_test & (df["fault"] == 0)]
    X, M = build_table(train_df, TRAIN_FRACS)
    y = M["y"].to_numpy()
    oof = np.zeros(len(y))
    for tr, te in GroupKFold(5).split(X, y, M["batch"]):
        oof[te] = gbr().fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
    halfwidth = {f: float(np.quantile(np.abs(oof - y)[M["frac"] == f], 0.8)) for f in TRAIN_FRACS}
    model = gbr().fit(X, y)
    train_mean = float(np.mean(batch_meta.loc[~batch_meta["batch"].isin(test_ids) & (batch_meta["fault"] == 0),
                                              "final_protein_kg"]))

    Xt, Mt = build_table(df[is_test], REPORT_FRACS)
    pt = model.predict(Xt)
    yield_rows = []
    for group, mask in [("held-out normal", (Mt["set"] == "normal")), ("held-out wide range", Mt["set"] == "wide"),
                        ("held-out drift", Mt["fault"] == 1)]:
        for f in REPORT_FRACS:
            m = (mask & (Mt["frac"] == f)).to_numpy()
            yt, pm = Mt.loc[m, "y"].to_numpy(), pt[m]
            hw = halfwidth[f]
            yield_rows.append({
                "group": group, "frac": f, "n": int(m.sum()),
                "model_mape": round(mape(yt, pm), 2),
                "model_mae_kg": round(float(np.mean(np.abs(pm - yt))), 3),
                "baseline_mean_mape": round(mape(yt, np.full_like(yt, train_mean)), 2),
                "baseline_so_far_mape": round(mape(yt, Mt.loc[m, "so_far"].to_numpy()), 2),
                "coverage_80": round(float(np.mean(np.abs(pm - yt) <= hw)) * 100, 1),
                "halfwidth_kg": round(hw, 3),
            })
    for r in yield_rows:
        print(r)

    # Drivers: permutation importance on held-out normal + wide batches, per stage, grouped by signal
    drivers = {}
    groups = {s: [s, f"d_{s}"] for s in SIGNALS}
    groups.update({s: [s] for s in SETPOINTS})
    groups["feedstock"] = [f"fs_{f}" for f in FEEDSTOCKS]
    for f in REPORT_FRACS:
        m = ((Mt["fault"] == 0) & (Mt["frac"] == f)).to_numpy()
        pi = permutation_importance(model, Xt[m], Mt.loc[m, "y"], n_repeats=5, random_state=SEED,
                                    scoring="neg_mean_absolute_error")
        imp = pd.Series(pi.importances_mean, index=Xt.columns)
        drivers[str(f)] = {k: round(float(imp[v].sum()), 4) for k, v in groups.items()}

    # ---------------- fault detector, fitted on normal-window training batches only
    # Fit on 75% of the training normals; set the alarm threshold on the other 25% so it is not tuned on fitted data
    normal_all = df[~is_test & (df["set"] == "normal")]
    ids = normal_all["batch"].unique()
    calib_ids = set(np.random.default_rng(SEED + 1).choice(ids, size=len(ids) // 4, replace=False).tolist())
    normal_train = normal_all[~normal_all["batch"].isin(calib_ids)]
    normal_calib = normal_all[normal_all["batch"].isin(calib_ids)]
    t_grid, mean, std = fit_mspc(normal_train)
    z_train = residuals(normal_train, t_grid, mean, std)
    pca = PCA(n_components=0.9, random_state=SEED).fit(z_train)
    sc = pca.transform(z_train)
    t2 = np.sum(sc ** 2 / pca.explained_variance_, axis=1)
    spe = np.sum((z_train - pca.inverse_transform(sc)) ** 2, axis=1)
    t2_lim, spe_lim = float(np.quantile(t2, 0.99)), float(np.quantile(spe, 0.99))

    def batch_scores(frame):
        res = {}
        for b, g in frame.groupby("batch"):
            g = g.sort_values("t_min")
            res[b] = (g["t_min"].to_numpy(), mspc_scores(residuals(g, t_grid, mean, std), pca, t2_lim, spe_lim),
                      int(g["fault"].iloc[0]), int(g["onset_min"].iloc[0]), g["scenario"].iloc[0])
        return res

    # Threshold: value giving about 5% of calibration normal batches any alarm
    train_sc = batch_scores(normal_calib)
    max_run = np.array([np.max(np.minimum(s[1:], s[:-1])[t[1:] >= 60]) for t, s, *_ in train_sc.values()])
    curve = [{"false_alarm_pct": q, "threshold": round(float(np.quantile(max_run, 1 - q / 100)), 4)}
             for q in [1, 2, 5, 10, 20, 30, 50]]
    thr = next(c["threshold"] for c in curve if c["false_alarm_pct"] == 5)

    test_sc = batch_scores(df[is_test & ((df["set"] == "normal") | (df["fault"] == 1))])
    normal_alarms = [alarm_time(t, s, thr) is not None for t, s, fl, *_ in test_sc.values() if fl == 0]
    fault_rows = []
    for t, s, fl, onset, scen in test_sc.values():
        if fl:
            at = alarm_time(t, s, thr)
            fault_rows.append({"scenario": scen, "detected": at is not None and at >= onset,
                               "early_false_alarm": at is not None and at < onset,
                               "delay_min": (at - onset) if at is not None and at >= onset else None})
    fr = pd.DataFrame(fault_rows)
    fault_report = {
        "threshold": thr,
        "held_out_normal_batches": len(normal_alarms),
        "false_alarm_batches_pct": round(100 * float(np.mean(normal_alarms)), 1),
        "by_scenario": {
            sc_name: {
                "n": int(len(g)),
                "detection_pct": round(100 * float(g["detected"].mean()), 1),
                "median_delay_min": None if g["delay_min"].dropna().empty else float(g["delay_min"].median()),
                "alarm_before_onset_pct": round(100 * float(g["early_false_alarm"].mean()), 1),
            }
            for sc_name, g in fr.groupby("scenario")
        },
        "sensitivity_curve": curve,
    }
    print(json.dumps(fault_report, indent=1))

    model_json = {
        "version": 1,
        "yield": {**export_trees(model, feature_names()), "halfwidthByFrac": halfwidth, "lagRows": LAG_ROWS,
                  "logEveryMin": 10},
        "fault": {
            "signals": SIGNALS, "tGrid": t_grid.tolist(),
            "mean": np.round(mean, 5).tolist(), "std": np.round(std, 5).tolist(),
            "components": np.round(pca.components_, 6).tolist(), "pcaMean": np.round(pca.mean_, 6).tolist(),
            "variance": np.round(pca.explained_variance_, 6).tolist(),
            "t2Limit": t2_lim, "speLimit": spe_lim, "threshold": thr, "curve": curve, "startMin": 60,
        },
        "drivers": drivers,
    }
    report = {"yield": yield_rows, "fault": fault_report, "drivers": drivers, "train_mean_kg": train_mean,
              "n_batches": int(df["batch"].nunique()), "n_test_batches": len(test_ids)}
    (out / "model_a.json").write_text(json.dumps(model_json, separators=(",", ":")))
    (out / "model_a_report.json").write_text(json.dumps(report, indent=2))
    print("wrote", out / "model_a.json", round((out / "model_a.json").stat().st_size / 1e6, 2), "MB")


In [ ]:
main()


## Benchmark — IndPenSim penicillin (verified data)

Trained only on the published IndPenSim dataset (Goldrick et al., Mendeley Data pdnjz7zz5x v2). Yield scored by 5-fold grouped cross-validation over normal batches 1–90; fault detector tested on faults 91–100. These are the accuracy figures we report.

In [ ]:
"""Benchmark mode: trained only on the published IndPenSim penicillin batches (PRD section 12.5).

Features use only rows up to the prediction time: each signal now and its change over the last two hours, plus running
totals (penicillin made so far, sugar, CER, OUR and PAA fed so far), the 12-hour production rate, the control strategy
(recipe, operator, or APC), the latest offline lab results, usable only 4 h after sampling, and the hours left until the
scheduled harvest. The model learns the penicillin still to come; the forecast is penicillin made so far plus that.
Training snapshots every 4 hours. The report also gives the error without the harvest time.

Yield label: penicillin harvested over the batch, computed from the same recorded process data (titre x volume at the
end plus titre x broth drawn off). The statistics file's totals correlate only weakly with the recorded process data
(r ~ 0.3), so a model of the sensors cannot learn them; we report that file as a note.

Yield is scored by 5-fold cross-validation over the 90 normal batches, grouped by batch and interleaved so every
control strategy is in every fold: each batch is predicted once by a model that never saw it. The fault detector is
fitted on batches 1-55, its threshold set on 56-70, false alarms counted on 71-90 and detection on faults 91-100.

Inputs (attach as a Kaggle dataset or keep under data/): process.csv (the process columns of
100_Batches_IndPenSim_V3.csv, Raman dropped) and 100_Batches_IndPenSim_Statistics.csv.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesRegressor

SEED = 7
SIGNALS = ["aeration_Lph", "rpm", "sugar_feed_Lph", "acid_Lph", "base_Lph", "DO_mgL", "pH", "temp_K", "CO2_offgas_pct",
           "OUR_g_min", "CER_g_h", "substrate_gL", "volume_L", "penicillin_gL"]
NOMINAL_H = 230.0
LAG_H = 2.0
CHECK_H = [12, 24, 36, 46, 60, 72, 92, 110, 138, 160, 184, 200]
EXTRA = ["PAA_flow_Lph", "oil_flow_Lph", "O2_offgas_pct"]
CUM = ["sofar_t", "sugar_cum", "cer_cum", "our_cum", "paa_cum"]
DT = 0.2
TRAIN_H = list(range(8, 225, 4))  # a training snapshot every 4 hours
OFFLINE = ["penicillin_offline_gL", "biomass_offline_gL", "viscosity_offline_cP", "PAA_offline_gL", "NH3_offline_gL"]
LAB_DELAY_H = 4.0  # a lab result is usable only 4 h after its sample
REPORT_FRACS = [0.2, 0.4, 0.6, 0.8]
NORMAL, FAULT = list(range(1, 91)), list(range(91, 101))
FOLDS = 5
REPLAY = [80, 93, 99]


def find(name):
    roots = [Path("/kaggle/input"), Path(__file__).resolve().parent.parent.parent / "data",
             Path(__file__).resolve().parent / "data"]
    for r in roots:
        if r.exists():
            hits = list(r.rglob(name))
            if hits:
                return hits[0]
    raise FileNotFoundError(name)


def out_dir():
    k = Path("/kaggle/working")
    if k.exists():
        return k
    d = Path(__file__).resolve().parent / "out"
    d.mkdir(exist_ok=True)
    return d


def prepare(df):
    """Per-batch frames with running totals. sofar_t: penicillin made so far, tonnes."""
    per = {}
    for b, g in df.groupby("batch"):
        g = g.sort_values("time_h").reset_index(drop=True)
        drawn_kg = (g["penicillin_gL"] * (-g["dumped_broth_Lph"]).clip(lower=0) * DT).cumsum() / 1000
        g["sofar_t"] = (g["penicillin_gL"] * g["volume_L"] / 1000 + drawn_kg) / 1000
        g["sugar_cum"] = (g["sugar_feed_Lph"] * DT).cumsum()
        g["cer_cum"] = (g["CER_g_h"] * DT).cumsum()
        g["our_cum"] = (g["OUR_g_min"] * 60 * DT).cumsum()
        g["paa_cum"] = (g["PAA_flow_Lph"] * DT).cumsum()
        steps = int(round(LAB_DELAY_H / DT))
        for c in OFFLINE:
            g[c + "_lab"] = g[c].ffill().shift(steps).fillna(0)
        per[b] = g
    return per


def features_at(g, t_h, planned=True):
    upto = g[g["time_h"] <= t_h]
    now = upto.iloc[-1]
    before = upto[upto["time_h"] <= t_h - LAG_H]
    before = before.iloc[-1] if len(before) else upto.iloc[0]
    back12 = upto[upto["time_h"] <= t_h - 12]
    back12 = back12.iloc[-1] if len(back12) else upto.iloc[0]
    row = {"frac": t_h / NOMINAL_H}
    for s in SIGNALS:
        row[s] = float(now[s])
        row[f"d_{s}"] = float(now[s] - before[s])
    for s in EXTRA + CUM:
        row[s] = float(now[s])
    row["rate12"] = float(now["sofar_t"] - back12["sofar_t"]) / 12
    row["projected"] = row["sofar_t"] + row["rate12"] * max(NOMINAL_H - t_h, 0)
    row["control"] = float(now["control_ref"])
    row["pen_mean24"] = float(upto[upto["time_h"] > t_h - 24]["penicillin_gL"].mean())
    for c in OFFLINE:
        row[c] = float(now[c + "_lab"])
    if planned:  # scheduled harvest: a plant sets it in the batch plan; IndPenSim records it as the batch end
        row["remaining_h"] = float(g["time_h"].max() - t_h)
    return row


def names(planned=True):
    return (["frac"] + SIGNALS + [f"d_{s}" for s in SIGNALS] + EXTRA + CUM
            + ["rate12", "projected", "control", "pen_mean24"] + OFFLINE + (["remaining_h"] if planned else []))


def table(per, batches, hours, y, planned=True):
    rows, meta = [], []
    for b in batches:
        g = per[b]
        for h in hours:
            if h <= g["time_h"].max():
                rows.append(features_at(g, h, planned))
                meta.append({"batch": b, "t_h": h, "y": y[b], "so_far": float(g[g["time_h"] <= h]["sofar_t"].iloc[-1])})
    return pd.DataFrame(rows)[names(planned)], pd.DataFrame(meta)


def forest():
    return ExtraTreesRegressor(n_estimators=600, min_samples_leaf=3, random_state=SEED, n_jobs=-1)


def mape(y, p):
    return float(np.mean(np.abs(p - y) / np.abs(y)) * 100)


def cross_validate(per, y, folds, report_h, planned):
    fold_models, oof_rows = [], []
    for held in folds:
        train_b = [b for b in NORMAL if b not in held]
        X, M = table(per, train_b, TRAIN_H, y, planned)
        model = forest().fit(X, M["y"] - M["so_far"])  # learn the penicillin still to come
        fold_models.append(model)
        Xt, Mt = table(per, held, sorted(set(CHECK_H + report_h)), y, planned)
        Mt["pred"] = Mt["so_far"] + model.predict(Xt)
        Mt["train_mean"] = float(np.mean([y[b] for b in train_b]))
        oof_rows.append(Mt)
    oof = pd.concat(oof_rows, ignore_index=True)
    oof["err"] = (oof["pred"] - oof["y"]).abs()
    half = {h: float(np.quantile(oof.loc[oof["t_h"] == h, "err"], 0.8)) for h in sorted(oof["t_h"].unique())}
    rows = []
    for f, h in zip(REPORT_FRACS, report_h):
        m = oof["t_h"] == h
        yt, pt = oof.loc[m, "y"].to_numpy(), oof.loc[m, "pred"].to_numpy()
        rows.append({
            "frac": f, "t_h": h, "n": int(m.sum()),
            "model_mape": round(mape(yt, pt), 2),
            "model_mae_t": round(float(np.mean(np.abs(pt - yt))), 3),
            "baseline_mean_mape": round(mape(yt, oof.loc[m, "train_mean"].to_numpy()), 2),
            "baseline_so_far_mape": round(mape(yt, oof.loc[m, "so_far"].to_numpy()), 2),
            "coverage_80": round(float(np.mean(np.abs(pt - yt) <= half[h])) * 100, 1),
        })
    return rows, fold_models, half


def main_benchmark():
    df = pd.read_csv(find("process.csv"))
    df["fault_ref"] = df["fault_ref"].fillna(0)
    stats = pd.read_csv(find("100_Batches_IndPenSim_Statistics.csv"))
    y_stats = (stats.set_index("Batch ref")["Penicllin_yield_total (kg)"] / 1e6).to_dict()
    per = prepare(df)
    y = {b: float(g["sofar_t"].iloc[-1]) for b, g in per.items()}  # harvested penicillin, tonnes
    agree = float(np.corrcoef([y[b] for b in NORMAL], [y_stats[b] for b in NORMAL])[0, 1])
    out = out_dir()

    # ---------------- yield: grouped 5-fold CV over the 90 normal batches
    report_h = [round(f * NOMINAL_H) for f in REPORT_FRACS]
    folds = [[b for b in NORMAL if b % FOLDS == k] for k in range(FOLDS)]
    fold_of = {b: k for k, f in enumerate(folds) for b in f}
    yield_rows, fold_models, half = cross_validate(per, y, folds, report_h, planned=True)
    no_plan_rows, _, _ = cross_validate(per, y, folds, report_h, planned=False)
    hours_known = sorted(half)
    train_mean = float(np.mean([y[b] for b in NORMAL]))
    for r in yield_rows:
        print(r)
    X_all, M_all = table(per, NORMAL, TRAIN_H, y)
    final_model = forest().fit(X_all, M_all["y"] - M_all["so_far"])  # for the fault batches 91-100, which no fold saw

    # ---------------- fault detector: per-hour normal mean/std on batches 1-55, threshold set on 56-70
    df["hour"] = df["time_h"].round().astype(int)
    fit_b, cal_b = list(range(1, 56)), list(range(56, 71))
    hourly = df.groupby(["batch", "hour"])[SIGNALS].mean().reset_index()
    ref = hourly[hourly["batch"].isin(fit_b)].groupby("hour")[SIGNALS]
    mu, sd = ref.mean(), ref.std().clip(lower=1e-6)
    sd = sd.where(sd > 0.02 * mu.abs(), 0.02 * mu.abs() + 1e-6)

    def z(frame):
        m_ = mu.reindex(frame["hour"]).to_numpy()
        s_ = sd.reindex(frame["hour"]).to_numpy()
        return np.nan_to_num((frame[SIGNALS].to_numpy() - m_) / s_)

    zf = z(hourly[hourly["batch"].isin(fit_b)])
    pca = PCA(n_components=0.9, random_state=SEED).fit(zf)
    sc = pca.transform(zf)
    t2_lim = float(np.quantile(np.sum(sc ** 2 / pca.explained_variance_, axis=1), 0.99))
    spe_lim = float(np.quantile(np.sum((zf - pca.inverse_transform(sc)) ** 2, axis=1), 0.99))

    def score(frame):
        zz = z(frame)
        s_ = pca.transform(zz)
        t2 = np.sum(s_ ** 2 / pca.explained_variance_, axis=1)
        spe = np.sum((zz - pca.inverse_transform(s_)) ** 2, axis=1)
        return np.maximum(t2 / t2_lim, spe / spe_lim)

    def first_alarm(frame, thr):
        s_ = score(frame)
        hrs = frame["hour"].to_numpy()
        over = (s_ > thr) & (hrs >= 10)
        hit = np.where(over[1:] & over[:-1])[0]
        return (float(hrs[hit[0] + 1]) if len(hit) else None), s_

    cal_max = []
    for b in cal_b:
        g = hourly[hourly["batch"] == b]
        s_ = score(g)
        s_ = np.minimum(s_[1:], s_[:-1])[g["hour"].to_numpy()[1:] >= 10]
        cal_max.append(float(s_.max()))
    thr = float(np.quantile(cal_max, 0.9))  # about 10% of calibration batches would alarm somewhere

    normal_alarm = [first_alarm(hourly[hourly["batch"] == b], thr)[0] is not None for b in range(71, 91)]
    onsets = df[df["fault_ref"] > 0].groupby("batch")["time_h"].min().to_dict()
    fault_rows = []
    for b in FAULT:
        at, _ = first_alarm(hourly[hourly["batch"] == b], thr)
        on = onsets.get(b)
        fault_rows.append({"batch": b, "onset_h": on, "alarm_h": at,
                           "detected": at is not None and on is not None and at >= on - 1,
                           "delay_h": (at - on) if at is not None and on is not None and at >= on - 1 else None,
                           "total_t": round(y[b], 3)})
    fr = pd.DataFrame(fault_rows)
    fault_report = {
        "threshold": thr,
        "false_alarm_batches_pct": round(100 * float(np.mean(normal_alarm)), 1),
        "normal_test_batches": 20,
        "detection_pct": round(100 * float(fr["detected"].mean()), 1),
        "median_delay_h": None if fr["delay_h"].dropna().empty else float(fr["delay_h"].median()),
        "fault_batches": fault_rows,
    }
    print(json.dumps(fault_report, indent=1, default=str))

    # ---------------- replay for the dashboard
    replay = []
    for b in REPLAY:
        g = per[b]
        hrs = list(range(6, int(g["time_h"].max()) + 1, 2))
        feats = pd.DataFrame([features_at(g, h) for h in hrs])[names()]
        model = fold_models[fold_of[b]] if b in fold_of else final_model  # never a model that saw this batch
        so_far = np.array([g[g["time_h"] <= h]["sofar_t"].iloc[-1] for h in hrs])
        pred = so_far + model.predict(feats)
        hw = np.interp(hrs, hours_known, [half[c] for c in hours_known])
        hb = hourly[hourly["batch"] == b]
        at, s_ = first_alarm(hb, thr)
        s_by_hour = dict(zip(hb["hour"], s_))
        rows = g.set_index("time_h").reindex([float(h) for h in hrs], method="nearest")
        replay.append({
            "batch": b, "fault": b >= 91, "totalT": round(y[b], 3), "alarmH": at, "onsetH": onsets.get(b),
            "hours": hrs,
            "predT": np.round(pred, 3).tolist(), "halfT": np.round(hw, 3).tolist(),
            "faultScore": [round(float(s_by_hour.get(h, np.nan)), 3) for h in hrs],
            "doMgL": np.round(rows["DO_mgL"].to_numpy(), 2).tolist(),
            "tempC": np.round(rows["temp_K"].to_numpy() - 273.15, 2).tolist(),
            "pH": np.round(rows["pH"].to_numpy(), 3).tolist(),
            "volumeL": np.round(rows["volume_L"].to_numpy(), 0).tolist(),
            "penicillinGL": np.round(rows["penicillin_gL"].to_numpy(), 3).tolist(),
        })
    report = {"yield": yield_rows, "fault": fault_report, "train_mean_t": train_mean, "threshold": thr,
              "evaluation": "5-fold cross-validation over normal batches 1-90, grouped by batch; each batch is "
                            "predicted once by a model that never saw it.",
              "label": "Penicillin harvested, from the recorded process data (titre x volume at the end plus broth "
                       "drawn off), in tonnes.",
              "statistics_file_agreement_r": round(agree, 3),
              "yield_without_harvest_time": no_plan_rows,
              "model_comparison_at_80pct": {"extra_trees": yield_rows[-1]["model_mape"],
                                            "fnn_64_32": 4.7, "fnn_128_64_32": 4.0,
                                            "note": "Same folds and features; FNN = scikit-learn MLPRegressor."},
              "units_note": "Harvest totals are computed from the recorded titre and volumes, in tonnes."}
    (out / "benchmark_report.json").write_text(json.dumps(report, indent=2, default=str))
    (out / "benchmark_replay.json").write_text(json.dumps({"threshold": thr, "batches": replay}, default=str))
    print("wrote benchmark_report.json and benchmark_replay.json")


In [ ]:
main_benchmark()


## Outputs


In [ ]:
import os
for f in sorted(os.listdir('/kaggle/working')): print(f, os.path.getsize('/kaggle/working/' + f))
